In [ ]:
# we make ad-hoc plots of the spectra in the 20250804 paramfiles.
# mostly this is cross spectra with ACT, and simple knox covariances.
# we also do a basic calibration to theory

import numpy as np
import matplotlib.pyplot as plt
from pspy import pspy_utils, so_spectra, so_dict
from pixell import enmap

import os
import re

lmin_for_cal = 1000
lmax_for_cal = 1800
lmax_for_plot = 3000

d = so_dict.so_dict()
d.read_from_file('/home/zatkins/projects/PSpipe_dev/repos/PSpipe/project/SO/pISO/paramfiles/tiger_dr6xbossn_20250804_coadd.dict')

spectra = ["TT", "TE", "TB", "ET", "BT", "EE", "EB", "BE", "BB"]

bin_low, bin_hi, bin_cent, bin_size = pspy_utils.read_binning_file(d['binning_file'], d['lmax'])
mask = bin_cent < lmax_for_plot

l_th, cmb_th = so_spectra.read_ps('/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_coadd/best_fits/dr6_lcdm_best_fits/cmb.dat', spectra)
assert l_th[0] == 2, "l_th should start at 2"
mask_th = l_th <= lmax_for_plot

In [ ]:
# get covariance of each cross-spectrum (knox)
# only care about spec autos

# get signal and noise power spectra
cmb_th_binned_dict = {spec: pspy_utils.naive_binning(l_th, cmb_th[spec], d['binning_file'], d['lmax'])[1] for spec in spectra}
raw_n_dr6_binned_dict = {}
raw_n_so_binned_dict = {}
n_dr6_binned_dict = {}
n_so_binned_dict = {}

for i, dr6_array1 in enumerate(d['arrays_dr6']):
    for dr6_array2 in d['arrays_dr6'][i:]:
        raw_n_dr6_binned_dict[dr6_array1, dr6_array2] = {}
        raw_n_dr6_binned_dict[dr6_array2, dr6_array1] = {}
        n_dr6_binned_dict[dr6_array1, dr6_array2] = {}
        n_dr6_binned_dict[dr6_array2, dr6_array1] = {}
        _, noise = so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_coadd/spectra/Dl_dr6_{dr6_array1}xdr6_{dr6_array2}_noise.dat', spectra)
        for spec in spectra:
            raw_n_dr6_binned_dict[dr6_array1, dr6_array2][spec] = noise[spec]
            raw_n_dr6_binned_dict[dr6_array2, dr6_array1][spec[::-1]] = raw_n_dr6_binned_dict[dr6_array1, dr6_array2][spec]

            n_dr6_binned_dict[dr6_array1, dr6_array2][spec] = raw_n_dr6_binned_dict[dr6_array1, dr6_array2][spec] # np.maximum(raw_n_dr6_binned_dict[dr6_array1, dr6_array2][spec], 0) if ((spec[0] == spec[1] and dr6_array1 == dr6_array2) or (spec == 'TT' and dr6_array1[2] == dr6_array2[2])) else np.zeros_like(bin_cent) # same array and autospec (or xfreq and TT)
            n_dr6_binned_dict[dr6_array2, dr6_array1][spec[::-1]] = n_dr6_binned_dict[dr6_array1, dr6_array2][spec]

for i, so_array1 in enumerate(d['arrays_so']):
    for so_array2 in d['arrays_so'][i:]:
        raw_n_so_binned_dict[so_array1, so_array2] = {}
        raw_n_so_binned_dict[so_array2, so_array1] = {}
        n_so_binned_dict[so_array1, so_array2] = {}
        n_so_binned_dict[so_array2, so_array1] = {}
        _, noise = so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_split/spectra/Dl_so_{so_array1}xso_{so_array2}_noise.dat', spectra)
        for spec in spectra:
            raw_n_so_binned_dict[so_array1, so_array2][spec] = noise[spec]
            raw_n_so_binned_dict[so_array2, so_array1][spec[::-1]] = raw_n_so_binned_dict[so_array1, so_array2][spec]

            n_so_binned_dict[so_array1, so_array2][spec] = raw_n_so_binned_dict[so_array1, so_array2][spec] # np.maximum(raw_n_so_binned_dict[so_array1, so_array2][spec], 0) if (spec[0] == spec[1] and so_array1[:2] == so_array2[:2]) else np.zeros_like(bin_cent) # same tube and autospec
            n_so_binned_dict[so_array2, so_array1][spec[::-1]] = n_so_binned_dict[so_array1, so_array2][spec]

In [ ]:
mask = bin_cent < 3500

for spec in spectra:
    for i, dr6_array1 in enumerate(d['arrays_dr6']):
        for dr6_array2 in d['arrays_dr6'][i:]:
            plt.plot(bin_cent[mask], n_dr6_binned_dict[dr6_array1, dr6_array2][spec][mask], label='fit')
            plt.plot(bin_cent[mask], raw_n_dr6_binned_dict[dr6_array1, dr6_array2][spec][mask], label='raw', zorder=0, alpha=0.5)
            plt.title(f'Noise {dr6_array1, dr6_array2, spec}')
            plt.legend()
            plt.axhline(0, color='k')
            plt.show()

for spec in spectra:
    for i, so_array1 in enumerate(d['arrays_so']):
        for so_array2 in d['arrays_so'][i:]:
            plt.plot(bin_cent[mask], n_so_binned_dict[so_array1, so_array2][spec][mask], label='fit')
            plt.plot(bin_cent[mask], raw_n_so_binned_dict[so_array1, so_array2][spec][mask], label='raw', zorder=0, alpha=0.5)
            plt.title(f'Noise {so_array1, so_array2, spec}')
            plt.legend()
            plt.axhline(0, color='k')
            plt.show()

In [ ]:
# get mask factors
dr6_masks = {}
so_masks = {}

for dr6_array in d['arrays_dr6']:
    dr6_masks[dr6_array] = enmap.read_map(d[f'window_T_dr6_{dr6_array}'])

for so_array in d['arrays_so']:
    so_masks[so_array] = enmap.read_map(d[f'window_T_so_{so_array}'])

w2_dict = {}
w4_dict = {}

for so_array in d['arrays_so']:
    w_so = so_masks[so_array]
    for dr6_array in d['arrays_dr6']:
        w_dr6 = dr6_masks[dr6_array]
        w2_dict[so_array, dr6_array] = np.sum(w_so * w_dr6 * w_so.pixsizemap()) / (4*np.pi)
        w2_dict[dr6_array, so_array] = w2_dict[so_array, dr6_array]

        for so_array2 in d['arrays_so']:
            for dr6_array2 in d['arrays_dr6']:
                key = tuple(sorted([so_array, so_array2, dr6_array, dr6_array2]))
                if key not in w4_dict:
                    w_so2 = so_masks[so_array2]
                    w_dr62 = dr6_masks[dr6_array2]
                    w4 = np.sum(w_so * w_dr6 * w_so2 * w_dr62 * w_so.pixsizemap()) / (4*np.pi)
                    w4_dict[key] = w4 

In [ ]:
# get cov
cov_dict = {spec: {} for spec in spectra}

for so_array in d['arrays_so']:
    for dr6_array in d['arrays_dr6']:
        for so_array2 in d['arrays_so']:
            for dr6_array2 in d['arrays_dr6']:
                w2 = w2_dict[so_array, dr6_array]
                w22 = w2_dict[so_array2, dr6_array2]
                w4_key = tuple(sorted([so_array, so_array2, dr6_array, dr6_array2]))
                w4 = w4_dict[w4_key]

                fac = w4/(w2*w22) / (2*bin_cent + 1) / bin_size

                for spec in spectra:
                    X, Y = spec
                    c = cmb_th_binned_dict[X+X]*cmb_th_binned_dict[Y+Y]
                    c += cmb_th_binned_dict[X+X]*n_so_binned_dict[so_array, so_array2][Y+Y]
                    c += n_dr6_binned_dict[dr6_array, dr6_array2][X+X]*cmb_th_binned_dict[Y+Y]
                    c += n_dr6_binned_dict[dr6_array, dr6_array2][X+X]*n_so_binned_dict[so_array, so_array2][Y+Y]
                    c += cmb_th_binned_dict[X+Y]*cmb_th_binned_dict[Y+X]

                    cov_dict[spec][(dr6_array, so_array), (dr6_array2, so_array2)] = fac * c

In [ ]:
def extract_names(filename):
    match = re.match(r'Dl_(.+)x(.+)_cross\.dat', filename)
    if match:
        return match.group(1), match.group(2)
    else:
        return None, None
    
def calibrate_data(lb, ps_data, ps_th, lmin, lmax, cov):
    mask = (lb >= lmin) & (lb <= lmax)
    num = np.sum(ps_data[mask] * ps_th[mask] / cov[mask])
    den = np.sum(ps_data[mask] ** 2 / cov[mask])
    return np.array([num / den, 1/den**0.5])

In [ ]:
cal_dict = {}
poleff_dict = {}

for so_array in d['arrays_so']:
    cmb_th_binned = {spec: 0 for spec in spectra}
    Dl_full = {spec: 0 for spec in spectra}
    Dl_gauss = {spec: 0 for spec in spectra}
    Dl_split = {spec: 0 for spec in spectra}
    cov = {spec: 0 for spec in spectra}
    for dr6_array in d['arrays_dr6']:
        # this is pretty accurate still even for the not-spin0xspin0 case
        Bbl = np.load(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/global_dr6xbossn_20250804_coadd/mcms/dr6_{dr6_array}xso_{so_array}_Bbl_spin0xspin0.npy')

        _, _Dl_full = so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/global_dr6xbossn_20250804_coadd/spectra/Dl_dr6_{dr6_array}xso_{so_array}_cross.dat', spectra)
        _, _Dl_gauss = so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/global_dr6xbossn_20250804_coadd_gauss_beam/spectra/Dl_dr6_{dr6_array}xso_{so_array}_cross.dat', spectra)
        _, _Dl_split = so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/global_dr6xbossn_20250804_split/spectra/Dl_dr6_{dr6_array}xso_{so_array}_cross.dat', spectra)
        for spec in spectra:
            cmb_th_binned[spec] += Bbl @ cmb_th[spec][:Bbl.shape[-1]] / len(d['arrays_dr6'])
            Dl_full[spec] += _Dl_full[spec] / len(d['arrays_dr6'])
            Dl_gauss[spec] += _Dl_gauss[spec] / len(d['arrays_dr6'])
            Dl_split[spec] += _Dl_split[spec] / len(d['arrays_dr6'])

            for dr6_array2 in d['arrays_dr6']:
                cov[spec] += cov_dict[spec][(dr6_array, so_array), (dr6_array2, so_array)] / len(d['arrays_dr6'])**2
    
    # get cal and poleff
    cal_full, e_cal_full = calibrate_data(bin_cent, Dl_full['TT'], cmb_th_binned['TT'], lmin_for_cal, lmax_for_cal, cov['TT'])
    poleff_full, inv_e_poleff_full = cal_full/calibrate_data(bin_cent, Dl_full['EE'], cmb_th_binned['EE'], lmin_for_cal, lmax_for_cal, cov['EE'])

    cal_dict[so_array] = cal_full
    poleff_dict[so_array] = poleff_full

    cal_gauss, e_cal_gauss = calibrate_data(bin_cent, Dl_gauss['TT'], cmb_th_binned['TT'], lmin_for_cal, lmax_for_cal, cov['TT'])
    poleff_gauss, inv_e_poleff_gauss = cal_gauss/calibrate_data(bin_cent, Dl_gauss['EE'], cmb_th_binned['EE'], lmin_for_cal, lmax_for_cal, cov['EE'])

    cal_split, e_cal_split = calibrate_data(bin_cent, Dl_split['TT'], cmb_th_binned['TT'], lmin_for_cal, lmax_for_cal, cov['TT'])
    poleff_split, inv_e_poleff_split = cal_split/calibrate_data(bin_cent, Dl_split['EE'], cmb_th_binned['EE'], lmin_for_cal, lmax_for_cal, cov['EE'])

    for spec in spectra:
        spec_so = spec[1]
        if spec_so == 'T':
            fac_full = cal_full
            fac_gauss = cal_gauss
            fac_split = cal_split
        else:
            fac_full = cal_full / poleff_full
            fac_gauss = cal_gauss / poleff_gauss
            fac_split = cal_split / poleff_split


        if spec == 'TT':
            label_full = f'cal, coadd, full beam transform=${cal_full:0.3f}\pm{e_cal_full:0.3f}$'
            label_gauss = f'cal, coadd, gaussian beam=${cal_gauss:0.3f}\pm{e_cal_gauss:0.3f}$'
            label_split = f'cal, split, full beam transform=${cal_split:0.3f}\pm{e_cal_split:0.3f}$'
        elif spec == 'EE':
            label_full = f'cal, coadd, full beam transform=${cal_full:0.3f}\pm{e_cal_full:0.3f}$\npoleff, coadd, full beam transform=${poleff_full:0.3f}\pm{1/inv_e_poleff_full:0.3f}$'
            label_gauss = f'cal, coadd, gaussian beam=${cal_gauss:0.3f}\pm{e_cal_gauss:0.3f}$\npoleff, coadd, gaussian beam=${poleff_gauss:0.3f}\pm{1/inv_e_poleff_gauss:0.3f}$'
            label_split = f'cal, split, full beam transform=${cal_split:0.3f}\pm{e_cal_split:0.3f}$\npoleff, split, full beam transform=${poleff_split:0.3f}\pm{1/inv_e_poleff_split:0.3f}$'
        else:
            label_full = None
            label_gauss = None
            label_split = None

        fig, axs = plt.subplots(1, 2, figsize=(12, 6), sharex=True)

        ax = axs[0]
        ax.errorbar(bin_cent[mask], fac_full * Dl_full[spec][mask], cov[spec][mask]**0.5, label='data, coadd, full beam transform', zorder=2)
        ax.errorbar(bin_cent[mask], fac_split * Dl_split[spec][mask], cov[spec][mask]**0.5, label='data, split, full beam transform', zorder=1)
        ax.errorbar(bin_cent[mask], fac_gauss * Dl_gauss[spec][mask], cov[spec][mask]**0.5, label='data, coadd, gaussian beam', alpha=0.5, zorder=0)
        ax.plot(l_th[mask_th], cmb_th[spec][mask_th], label='theory', linestyle='--', color='k', alpha=0.3)
        ax.set_title(f'avg_ar(ACT_ar x {so_array}), {spec}')
        ax.legend()
        ax.set_xlabel('l')
        ax.set_ylabel(f'Dl [uk^2]')

        ax = axs[1]
        if spec[0] == spec[1]:
            ylabel = 'data / theory'
            ax.errorbar(bin_cent[mask], fac_full * Dl_full[spec][mask] / cmb_th_binned[spec][mask], cov[spec][mask]**0.5 / cmb_th_binned[spec][mask], label=label_full, zorder=2)
            ax.errorbar(bin_cent[mask], fac_split * Dl_split[spec][mask] / cmb_th_binned[spec][mask], cov[spec][mask]**0.5 / cmb_th_binned[spec][mask], label=label_split, zorder=1)
            ax.errorbar(bin_cent[mask], fac_gauss * Dl_gauss[spec][mask] / cmb_th_binned[spec][mask], cov[spec][mask]**0.5 / cmb_th_binned[spec][mask], alpha=0.5, zorder=0, label=label_gauss)
            ax.axhline(1, linestyle='--', color='k', alpha=0.3)
            if spec == 'TT':
                ax.set_ylim(.75, 1.25)
            else:
                ax.set_ylim(.25, 1.75)
        else:
            ylabel = 'data - theory'
            ax.errorbar(bin_cent[mask], fac_full * Dl_full[spec][mask] - cmb_th_binned[spec][mask], cov[spec][mask]**0.5, label=label_full, zorder=2)
            ax.errorbar(bin_cent[mask], fac_split * Dl_split[spec][mask] - cmb_th_binned[spec][mask], cov[spec][mask]**0.5, label=label_split, zorder=1)
            ax.errorbar(bin_cent[mask], fac_gauss * Dl_gauss[spec][mask] - cmb_th_binned[spec][mask], cov[spec][mask]**0.5, alpha=0.5, zorder=0, label=label_gauss)
            ax.axhline(0, linestyle='--', color='k', alpha=0.3)
        ax.set_title(f'avg_ar(ACT_ar x {so_array}) vs. theory, {spec}')
        if spec == 'TT' or spec == 'EE':
            ax.legend()
        ax.set_xlabel('l')
        ax.set_ylabel(ylabel)
        ax.grid()

        ax.set_xlim(0, lmax_for_plot)
        
        plt.tight_layout()
        plt.show()

In [ ]:
so_array = 'i1_f090'

lmin_for_plot = 600
lmax_for_plot = 2500
mask = (lmin_for_plot <= bin_cent) & (bin_cent <= lmax_for_plot)
mask_th = (lmin_for_plot <= l_th) & (l_th <= lmax_for_plot)

spec = 'TT'

cov_facs = [1, 2]

for cov_fac in cov_facs:
    fig, ax = plt.subplots(figsize=(12, 6))

    for i, dr6_arrays in enumerate([['pa5_f090', 'pa6_f090'], ['pa5_f150', 'pa6_f150']]):
        Dl = 0
        cov = 0
        lscale = 2 if spec == 'TT' else 0
        yscale = 1e6 if spec == 'TT' else 1
        for dr6_array in dr6_arrays:
            Dl += so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/global_dr6xbossn_20250804_coadd/spectra/Dl_dr6_{dr6_array}xso_{so_array}_cross.dat', spectra)[1][spec] / len(dr6_arrays)
            for dr6_array2 in dr6_arrays:
                cov += cov_dict[spec][(dr6_array, so_array), (dr6_array2, so_array)] / len(dr6_arrays)**2
        cov *= cov_fac

        label = ['SO i1 f090 x ACT f090', 'SO i1 f090 x ACT f150'][i]
        ax.errorbar(bin_cent[mask] + (i - 0.5*len(dr6_arrays))/len(dr6_arrays) * 25, Dl[mask] * bin_cent[mask]**lscale / yscale, cov[mask]**0.5 * bin_cent[mask]**lscale  / yscale, linestyle='none', fmt='.', markerfacecolor='white', label=label)

    ax.plot(l_th[mask_th], cmb_th[spec][mask_th] * l_th[mask_th]**lscale / yscale, color='k', alpha=0.5, linestyle='--', zorder=0, label='ACT DR6 LCDM (CMB-only)')
    ax.set_xlim(lmin_for_plot, lmax_for_plot)
    ax.set_xlabel(r'$\ell$', fontsize=16)
    ax.set_ylabel(fr'$\ell^{{{lscale}}}D_\ell^{{TT}}\ \rm [mK^2]$', fontsize=16)

    order = (1, 2, 0)
    handles, labels = ax.get_legend_handles_labels()
    h = [handles[i] for i in order]
    l = [labels[i] for i in order]
    ax.legend(h, l, fontsize=16)
    plt.show()

specs = ['ET', 'TE']
for cov_fac in cov_facs:
    fig, ax = plt.subplots(figsize=(12, 6))

    for i, spec in enumerate(specs):
        Dl = 0
        cov = 0
        for dr6_array in d['arrays_dr6']:
            Dl += so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/global_dr6xbossn_20250804_coadd/spectra/Dl_dr6_{dr6_array}xso_{so_array}_cross.dat', spectra)[1][spec] / len(d['arrays_dr6'])
            for dr6_array2 in d['arrays_dr6']:
                cov += cov_dict[spec][(dr6_array, so_array), (dr6_array2, so_array)] / len(d['arrays_dr6'])**2
        cov *= cov_fac

        label = ['SO i1 f090 x ACT (all arrays)', 'ACT (all arrays) x SO i1 f090'][i]
        ax.errorbar(bin_cent[mask] + (i - 0.5*len(specs))/len(specs) * 25, Dl[mask], cov[mask]**0.5, linestyle='none', fmt='.', markerfacecolor='white', label=label)

    ax.plot(l_th[mask_th], cmb_th[spec][mask_th], color='k', alpha=0.5, linestyle='--', zorder=0, label='ACT DR6 LCDM (CMB-only)')
    ax.set_xlim(lmin_for_plot, lmax_for_plot)
    ax.set_xlabel(r'$\ell$', fontsize=16)
    ax.set_ylabel(r'$D_\ell^{TE}\ \rm [\mu K^2]$', fontsize=16)

    order = (1, 2, 0)
    handles, labels = ax.get_legend_handles_labels()
    h = [handles[i] for i in order]
    l = [labels[i] for i in order]
    ax.legend(h, l, fontsize=16)
    plt.show()

spec = 'EE'
for cov_fac in cov_facs:
    fig, ax = plt.subplots(figsize=(12, 6))

    Dl = 0
    cov = 0
    for dr6_array in d['arrays_dr6']:
        Dl += so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/global_dr6xbossn_20250804_coadd/spectra/Dl_dr6_{dr6_array}xso_{so_array}_cross.dat', spectra)[1][spec] / len(d['arrays_dr6'])
        for dr6_array2 in d['arrays_dr6']:
            cov += cov_dict[spec][(dr6_array, so_array), (dr6_array2, so_array)] / len(d['arrays_dr6'])**2
    cov *= cov_fac

    label = 'SO i1 f090 x ACT (all arrays)'
    ax.errorbar(bin_cent[mask], Dl[mask], cov[mask]**0.5, linestyle='none', fmt='.', markerfacecolor='white', label=label)
    ax.plot(l_th[mask_th], cmb_th[spec][mask_th], color='k', alpha=0.5, linestyle='--', zorder=0, label='ACT DR6 LCDM (CMB-only)')
    ax.set_xlim(lmin_for_plot, lmax_for_plot)
    ax.set_xlabel(r'$\ell$', fontsize=16)
    ax.set_ylabel(r'$D_\ell^{EE}\ \rm [\mu K^2]$', fontsize=16)
    ax.legend(fontsize=16)

    order = (1, 0)
    handles, labels = ax.get_legend_handles_labels()
    h = [handles[i] for i in order]
    l = [labels[i] for i in order]
    ax.legend(h, l, fontsize=16)
    plt.show()

In [ ]:
## EE variations, fac of 2 in cov only

so_array = 'i1_f090'

lmin_for_plot = 600
lmax_for_plot = 2500
mask = (lmin_for_plot <= bin_cent) & (bin_cent <= lmax_for_plot)
mask_th = (lmin_for_plot <= l_th) & (l_th <= lmax_for_plot)

spec = 'EE'
cov_fac = 2

Dl = 0
cov = 0
for dr6_array in d['arrays_dr6']:
    Dl += so_spectra.read_ps(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/tiger_dr6xbossn_20250804_coadd/spectra/Dl_dr6_{dr6_array}xso_{so_array}_cross.dat', spectra)[1][spec] / len(d['arrays_dr6'])
    for dr6_array2 in d['arrays_dr6']:
        cov += cov_dict[spec][(dr6_array, so_array), (dr6_array2, so_array)] / len(d['arrays_dr6'])**2
cov *= cov_fac

fig, ax = plt.subplots(figsize=(12, 6))

label = 'SO i1 f090 x ACT (all arrays)'
ax.errorbar(bin_cent[mask], Dl[mask], cov[mask]**0.5, linestyle='none', fmt='.', markerfacecolor='white', label=label)
ax.plot(l_th[mask_th], cmb_th[spec][mask_th], color='k', alpha=0.5, linestyle='--', zorder=0, label='ACT DR6 LCDM (CMB-only)')
ax.set_xlim(lmin_for_plot, lmax_for_plot)
ax.set_xlabel(r'$\ell$', fontsize=16)
ax.set_ylabel(r'$D_\ell^{EE}\ \rm [\mu K^2]$', fontsize=16)
ax.legend(fontsize=16)

order = (1, 0)
handles, labels = ax.get_legend_handles_labels()
h = [handles[i] for i in order]
l = [labels[i] for i in order]
ax.legend(h, l, fontsize=16)

ax.text(0.5, 0.75, 'PRELIMINARY', transform=ax.transAxes, fontsize=48, color='red', verticalalignment='top')

plt.show()

fig, ax = plt.subplots(figsize=(12, 6))

label = 'SO i1 f090 x ACT (all arrays)'
ax.plot(bin_cent[mask], Dl[mask], marker='.', markersize=12, linestyle='none', markerfacecolor='white', label=label)
ax.plot(l_th[mask_th], cmb_th[spec][mask_th], color='k', alpha=0.5, linestyle='--', zorder=0, label='ACT DR6 LCDM (CMB-only)')
ax.set_xlim(lmin_for_plot, lmax_for_plot)
ax.set_xlabel(r'$\ell$', fontsize=16)
ax.set_ylabel(r'$D_\ell^{EE}\ \rm [\mu K^2]$', fontsize=16)
ax.legend(fontsize=16)

order = (0, 1) # (1, 0)
handles, labels = ax.get_legend_handles_labels()
h = [handles[i] for i in order]
l = [labels[i] for i in order]
ax.legend(h, l, fontsize=16)

ax.text(0.5, 0.75, 'PRELIMINARY', transform=ax.transAxes, fontsize=48, color='red', verticalalignment='top')

plt.show()

fig, ax = plt.subplots(figsize=(12, 6))

label = 'SO i1 f090 x ACT (all arrays)'
ax.errorbar(bin_cent[mask], Dl[mask], cov[mask]**0.5, linestyle='none', fmt='.', markerfacecolor='white', label=label)
ax.plot(l_th[mask_th], cmb_th[spec][mask_th], color='k', alpha=0.5, linestyle='--', zorder=0, label='ACT DR6 LCDM (CMB-only)')
ax.set_xlim(lmin_for_plot, lmax_for_plot)
ax.set_xlabel(r'$\ell$', fontsize=16)
ax.set_ylabel(r'$D_\ell^{EE}$', fontsize=16) # ax.set_ylabel(r'$D_\ell^{EE}\ \rm [\mu K^2]$', fontsize=16)
ax.set_yticks([])
ax.legend(fontsize=16)

order = (1, 0)
handles, labels = ax.get_legend_handles_labels()
h = [handles[i] for i in order]
l = [labels[i] for i in order]
ax.legend(h, l, fontsize=16)

ax.text(0.5, 0.75, 'PRELIMINARY', transform=ax.transAxes, fontsize=48, color='red', verticalalignment='top')

plt.show()